# テーマ3 回帰分析

## 目的
- 回帰が「入力から連続値を予測する問題」であることを理解する
- California Housing データで、特徴量と住宅価格の関係を散布図と相関係数で確認する
- train/test 分割を使い、未知データでモデルを評価する
- MAE、RMSE、予測値と実測値の散布図から結果を解釈する

## 今日の成果物
- 価格と関係がありそうな特徴量を、グラフと相関係数から1つ挙げる
- 回帰モデルの誤差を表で確認する
- 予測値と実測値の図から、モデルの当てはまりを短く説明する

## 注意
- California Housing は初回にダウンロードが必要な場合があります
- セルは上から順に実行してください


## 回帰の基本

回帰は、入力データから連続値を予測する問題です。例えば、住宅の情報から価格を予測する場合、入力 $X$ と目的変数 $y$ の関係をモデルで近似します。

線形回帰では、予測値 $\hat{y}$ を次のように表します。

$$
\hat{y} = w_1x_1 + w_2x_2 + \cdots + w_px_p + b
$$

ここで、$x_1, x_2, \ldots, x_p$ は特徴量、$w_1, w_2, \ldots, w_p$ は重み、$b$ は切片です。モデルは、予測値と実測値のずれが小さくなるように重みを学習します。


### Step 0: ライブラリを読み込む
回帰分析で使うライブラリを準備します。線形回帰、決定木、ランダムフォレストを同じ条件で比較できるようにします。


In [ ]:
# ===== Step 0: ライブラリ読み込み =====
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

plt.style.use('seaborn-v0_8')


### Step 1: データを読み込む
California Housing データを読み込み、回帰の目的変数を確認します。最初にデータの形と先頭行を見て全体像をつかみます。


In [ ]:
# ===== Step 1: データ読み込み =====
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()
target_column = 'MedHouseVal'

print('行数, 列数 =', df.shape)
display(df.head())
display(df.describe())


### Step 2: 散布図で関係を見る
説明変数と目的変数の関係を散布図で確認します。「どの変数が効いていそうか」「外れた値はあるか」「直線的な関係に見えるか」に注目してください。


#### 観察ポイント

散布図を見たら、次の問いに答えてください。

1. 点は右上がり、右下がり、ばらばらのどれに近いですか。
2. 極端に外れた点はありますか。
3. 直線だけで説明できそうですか。
4. 価格が上限で止まっているような不自然な形はありますか。

モデルを作る前の観察は、あとで誤差や失敗例を解釈する助けになります。


In [ ]:
# ===== Step 2: 散布図で関係を見る =====
# TODO: x_col を別の特徴量に変えて試す
x_col = 'MedInc'
y_col = target_column

plt.figure(figsize=(6, 4))
plt.scatter(df[x_col], df[y_col], alpha=0.2)
plt.xlabel(x_col)
plt.ylabel(y_col)
plt.title(f'{x_col} と {y_col} の散布図')
plt.show()


### Step 3: 相関係数で関係の強さを見る
回帰では、目的変数と関係がありそうな特徴量を探すことが重要です。散布図で見た関係を、相関係数でも確認します。

相関係数は、2つの変数がどの程度一緒に動くかを表す数値です。

$$
r = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_i (x_i - \bar{x})^2}\sqrt{\sum_i (y_i - \bar{y})^2}}
$$

値の目安は次の通りです。

- $r \simeq 1$: 強い正の相関
- $r \simeq -1$: 強い負の相関
- $r \simeq 0$: 直線的な関係は弱い

ただし、相関があることは「原因である」ことを意味しません。相関と因果は区別して考えます。


In [ ]:
# ===== Step 3: 目的変数との相関を確認する =====
# TODO: 目的変数 MedHouseVal と関係が強そうな特徴量を確認する
corr_with_target = df.corr(numeric_only=True)[target_column].sort_values(ascending=False)
display(corr_with_target)

plt.figure(figsize=(7, 4))
corr_with_target.drop(target_column).sort_values().plot(kind='barh')
plt.xlabel('相関係数')
plt.title(f'{target_column} との相関')
plt.show()


In [ ]:
# ===== 追加演習: 相関行列を可視化する =====
# TODO: feature_columns_for_corr を変えて、見たい特徴量を選ぶ
feature_columns_for_corr = ['MedInc', 'AveRooms', 'HouseAge', 'AveOccup', target_column]
corr_matrix = df[feature_columns_for_corr].corr(numeric_only=True)
display(corr_matrix)

plt.figure(figsize=(6, 5))
plt.imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(label='相関係数')
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=45, ha='right')
plt.yticks(range(len(corr_matrix.index)), corr_matrix.index)
plt.title('相関行列')
plt.tight_layout()
plt.show()


### Step 4: 学習用とテスト用に分ける
モデルが学習に使ったデータだけに合っているのか、未知データでも使えるのかを分けて確認するため、train と test に分割します。


#### 解説: train/test 分割

同じデータで学習と評価を行うと、モデルが「覚えただけ」でも良い点数に見えてしまいます。そこで、データを分けます。

- train: モデルが学習に使うデータ
- test: 学習には使わず、最後に評価するデータ

この考え方は、未知のデータに対してどれくらい通用するかを確認するために重要です。


#### 解説: MAE と RMSE

予測値を $\hat{y}_i$、実測値を $y_i$ とすると、誤差は $y_i - \hat{y}_i$ です。

MAE は、誤差の絶対値の平均です。

$$
MAE = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|
$$

RMSE は、誤差を2乗して平均し、最後に平方根を取ります。

$$
RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}
$$

RMSE は大きな誤差をより強く反映します。MAE と RMSE のどちらも、値が小さいほど予測と実測のずれが小さいと考えます。


In [ ]:
# ===== 追加演習: 誤差を手計算のイメージで確認する =====
example_actual = pd.Series([3.0, 4.0, 5.0])
example_pred = pd.Series([2.5, 4.5, 4.0])
example_error = example_actual - example_pred

print('誤差:', example_error.tolist())
print('絶対誤差:', example_error.abs().tolist())
print('MAE:', example_error.abs().mean())
print('RMSE:', (example_error.pow(2).mean()) ** 0.5)


In [ ]:
# ===== Step 4: 学習データとテストデータを作る =====
feature_columns = ['MedInc', 'AveRooms', 'HouseAge', 'AveOccup']  # TODO: 必要に応じて変更する
X = df[feature_columns]
y = df[target_column]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('X_train shape =', X_train.shape)
print('X_test shape =', X_test.shape)


### Step 5: 複数の回帰モデルを比較する
線形回帰、決定木、ランダムフォレストを同じデータで学習し、MAE と RMSE を比べます。誤差は小さいほど予測と実測のずれが小さいことを表します。


In [ ]:
# ===== Step 5: モデル比較 =====
models = {
    'linear_regression': LinearRegression(),
    'decision_tree': DecisionTreeRegressor(random_state=42, max_depth=5),
    'random_forest': RandomForestRegressor(random_state=42, n_estimators=100, max_depth=8),
}

results = []
predictions_by_model = {}

for model_name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    predictions_by_model[model_name] = preds
    results.append({
        'model': model_name,
        'mae': mean_absolute_error(y_test, preds),
        'rmse': mean_squared_error(y_test, preds, squared=False),
    })

results_df = pd.DataFrame(results).sort_values('rmse')
display(results_df)


### Step 6: 予測値と実測値を可視化する
選んだモデルの予測値と実測値を散布図で比べます。理想的には、点が赤い対角線の近くに集まります。


In [ ]:
# ===== Step 6: 予測値と実測値の可視化 =====
# TODO: selected_model を変えて比較する
selected_model = results_df.iloc[0]['model']
selected_predictions = predictions_by_model[selected_model]

plt.figure(figsize=(6, 6))
plt.scatter(y_test, selected_predictions, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red')
plt.xlabel('実測値')
plt.ylabel('予測値')
plt.title(f'{selected_model} の予測 vs 実測')
plt.show()


## 発展演習

時間があれば、次を試してください。

1. `feature_columns` に `Latitude` や `Longitude` を追加する。
2. `DecisionTreeRegressor` の `max_depth` を変える。
3. `RandomForestRegressor` の `n_estimators` を変える。
4. RMSE が小さくなったとき、散布図の見え方も良くなっているか確認する。

数値が良くなっただけで終わらず、図と合わせて説明してください。


## 振り返り
- 目的変数との相関が強かった特徴量はどれですか。散布図の見え方と一致していましたか。
- `feature_columns` を増やしたり減らしたりすると、誤差はどう変わりましたか。
- どのモデルの RMSE が最も小さかったですか。
- 予測値と実測値の散布図から、どのような失敗や限界が見えましたか。
- train/test 分割を行う目的を、自分の言葉で説明してください。
